In [5]:
text_a = open("1342-0.txt", encoding="utf-8").read()
text_b = open("84-0.txt", encoding="utf-8").read()
print('Loaded text_a (chars:', len(text_a), ') and text_b (chars:', len(text_b), ')')

Loaded text_a (chars: 704189 ) and text_b (chars: 440747 )


# Simple Markov-chain Text Generator
This notebook implements a simple word-level (order-1) Markov chain text generator.
- Build a model mapping each token to counts of following tokens.
- Generate text by sampling next tokens from the learned distribution.
Usage: run the implementation cell, then call `generate_markov(...)` with `length` and optional `seed`.

In [6]:
from collections import defaultdict, Counter
import random
import re

def tokenize_words(text):
    # keep words and punctuation as separate tokens
    return re.findall(r"\w+|[^\w\s]", text, re.UNICODE)

def build_markov_model(text, order=1):
    tokens = tokenize_words(text)
    model = defaultdict(Counter)
    if len(tokens) <= order:
        return {}
    for i in range(len(tokens) - order):
        key = tuple(tokens[i:i+order])
        next_tok = tokens[i+order]
        model[key][next_tok] += 1
    return {k: dict(v) for k, v in model.items()}

def _weighted_choice(choices_dict):
    tokens, weights = zip(*choices_dict.items())
    total = sum(weights)
    probs = [w/total for w in weights]
    return random.choices(tokens, probs)[0]

def generate_markov(model, length=50, order=1, seed=None):
    if not model:
        return ""
    # choose initial seed
    if seed is None:
        seed = random.choice(list(model.keys()))
    else:
        if isinstance(seed, str):
            seed_tokens = tokenize_words(seed)
            seed = tuple(seed_tokens[-order:]) if seed_tokens else random.choice(list(model.keys()))
        else:
            seed = tuple(seed)
    output = list(seed)
    for _ in range(length):
        key = tuple(output[-order:])
        choices = model.get(key)
        if not choices:
            # fallback to a random key if current key has no continuation
            key = random.choice(list(model.keys()))
            choices = model[key]
        next_tok = _weighted_choice(choices)
        output.append(next_tok)
    # join tokens with spacing rules: put space before word tokens, no extra space before punctuation
    out = []
    for tok in output:
        if out and re.match(r"\w", tok):
            out.append(' ' + tok)
        else:
            out.append(tok)
    return ''.join(out)

In [7]:
# Build model from available texts and generate examples
combined_text = text_a + '\n' + text_b
model = build_markov_model(combined_text, order=1)

# Generate three short examples with different seeds (None = random)
print('--- Example 1 (random seed) ---')
print(generate_markov(model, length=80, order=1, seed=None))
print('\n--- Example 2 (seed: first words of text_a) ---')
seed = ''
print(generate_markov(model, length=80, order=1, seed=seed))
print('\n--- Example 3 (seed: a short phrase) ---')
print(generate_markov(model, length=80, order=1, seed='The'))

--- Example 1 (random seed) ---
rare instances of nothing, and, Elizabeth on Elizabeth allowed myself. The ball at no longer able to become acquainted with more than your bottle of devilish despair and the youth and aunt, my own have exchanged the dance on both to know very impertinent and his hat- shire might be of happiness, Esq. Thoughtless and, as either seeing Lady Catherine was with pleasure. They saw the next morning at the country or

--- Example 2 (seed: first words of text_a) ---
disbelieving the compliment deserved. Sometimes I dare say in the knowledge. And what manner; I believe and three months that your path, if he entered, after slightly colouring with ardour. Elizabeth Bennet, and wrapping myself to more concentrated disposition, and at having done very proud and to him out my part of letters which I have been, have done- extinguished every day arrived which the first opened the students, I

--- Example 3 (seed: a short phrase) ---
The gentlemen, and ambition of elder 